[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/service_assistant/plain_python.ipynb)

# Simulated service assistant — plain Python

Exact-action approval, least privilege, idempotency, interruption, checkpoint/resume and recovery. No external service is contacted. Mock runtime is under one minute on CPU.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import json
from pathlib import Path
from typing import ClassVar

from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture = json.loads((ROOT / "data/service_assistant/case_mock.json").read_text())

## Visible approval and recovery workflow
The model proposes an action but cannot execute it. Python checks action name, permission scope and an approval payload tied to every exact argument, checkpoints before the effect, then uses the idempotency key to make retries safe.

In [ ]:
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: str
    order_id: str
    new_address: str
    idempotency_key: str
    requires_approval: bool


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: str


def model():
    return DeterministicMockClient(ScriptedScenarioFixture.model_validate(fixture))


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


async def run_service():
    client = model()
    service = SimulatedService.from_path(service_path)
    trace = []
    before = service.read_order("ord-100", actor="tutorial-user")
    trace.append({"event": "read", "address": before["delivery_address"]})
    action = await propose(client, Action, "Propose the requested address update.")
    allowed = action.action == "update_address" and action.requires_approval
    approval = {
        "order_id": "ord-100",
        "new_address": "2 Evidence Road",
        "idempotency_key": "address-ord-100-v1",
    }
    exact = approval == {
        "order_id": action.order_id,
        "new_address": action.new_address,
        "idempotency_key": action.idempotency_key,
    }
    if not allowed or not exact:
        return {"outcome": "refused", "trace": [*trace, {"event": "approval_denied"}]}
    checkpoint = service.checkpoint()
    trace.append({"event": "interrupted_after_checkpoint"})
    service = SimulatedService.resume(checkpoint)
    receipt = service.update_address(
        action.order_id,
        action.new_address,
        actor="tutorial-user",
        idempotency_key=action.idempotency_key,
    )
    duplicate = service.replay(action.idempotency_key)
    trace.extend(
        [
            {"event": "effect", "receipt": receipt},
            {"event": "duplicate_detected", "duplicate": duplicate["duplicate"]},
        ]
    )
    confirmation = await propose(client, Confirmation, f"Confirm this receipt only: {receipt}")
    return {
        "outcome": confirmation.status,
        "receipt": receipt,
        "duplicate": duplicate,
        "trace": trace,
        "termination": "criteria_met",
        "usage": {"model_calls": 2, "tool_calls": 2},
    }


first = await run_service()
second = await run_service()
evaluation = {
    "component": first["receipt"]["delivery_address"] == "2 Evidence Road",
    "trajectory": len(first["trace"]) == 4,
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first == second,
}
assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "result": first,
    "resource_report": first["usage"],
    "fallback": "refuse or escalate when scope or exact approval differs",
}

## Limitations
The environment is intentionally simulated; real services require durable transactional storage, authenticated approvers and provider-specific failure handling.